# 🏭 Predictive Maintenance System for Industrial Pump Failure

### Machine Learning Pipeline
This notebook demonstrates the complete ML pipeline for predicting industrial pump failures using sensor data.

**Features analyzed:**
- Temperature (°C)
- Vibration (mm/s)
- Pressure (bar)
- Flow Rate (L/min)
- RPM
- Power Consumption (kW)
- Humidity (%)
- Noise Level (dB)

**Models compared:**
- Logistic Regression
- Random Forest
- Support Vector Machine (SVM)
- Gradient Boosting

---

## 1️⃣ Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)
import warnings
warnings.filterwarnings('ignore')

# Style configuration
plt.style.use('dark_background')
sns.set_palette('muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.facecolor'] = '#1a1f35'
plt.rcParams['figure.facecolor'] = '#0a0e1a'
plt.rcParams['text.color'] = '#e4e7f1'
plt.rcParams['axes.labelcolor'] = '#8892a8'
plt.rcParams['xtick.color'] = '#8892a8'
plt.rcParams['ytick.color'] = '#8892a8'

print('✅ Libraries loaded successfully!')

## 2️⃣ Generate Synthetic Sensor Data

In [ ]:
from data.generate_data import generate_pump_sensor_data, save_data

# Generate 5000 samples with 15% failure rate
df = generate_pump_sensor_data(n_samples=5000, failure_ratio=0.15)
save_data(df)

print(f"\n📊 Dataset Shape: {df.shape}")
print(f"📋 Columns: {list(df.columns)}")
df.head(10)

## 3️⃣ Exploratory Data Analysis (EDA)

In [ ]:
# Basic statistics
print("📈 Dataset Statistics:")
df.describe().round(2)

In [ ]:
# Check for missing values
print("🔍 Missing Values:")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.DataFrame({'Missing Count': missing, 'Percentage (%)': missing_pct}))

# Class distribution
print(f"\n⚙️ Class Distribution:")
print(df['failure'].value_counts())
print(f"\nFailure Rate: {df['failure'].mean():.2%}")

In [ ]:
# Failure distribution by pump
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
colors = ['#38ef7d', '#f5576c']
labels = ['Normal', 'Failure']
counts = df['failure'].value_counts().sort_index()
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=0.5, width=0.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold', fontsize=12, color='#e4e7f1')
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold', color='#e4e7f1')
axes[0].set_ylabel('Count')

# Failure rate by pump
pump_failure = df.groupby('pump_id')['failure'].mean() * 100
pump_failure.plot(kind='bar', ax=axes[1], color='#667eea', edgecolor='white', linewidth=0.5)
axes[1].set_title('Failure Rate by Pump (%)', fontsize=14, fontweight='bold', color='#e4e7f1')
axes[1].set_ylabel('Failure Rate (%)')
axes[1].set_xlabel('Pump ID')
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(pump_failure.values):
    axes[1].text(i, v + 0.3, f'{v:.1f}%', ha='center', fontsize=10, color='#e4e7f1')

plt.tight_layout()
plt.savefig('data/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature distributions: Normal vs Failure
features = ['temperature_C', 'vibration_mm_s', 'pressure_bar', 'flow_rate_L_min',
             'rpm', 'power_kW', 'humidity_pct', 'noise_level_dB']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, feature in enumerate(features):
    normal_data = df[df['failure'] == 0][feature].dropna()
    failure_data = df[df['failure'] == 1][feature].dropna()
    
    axes[i].hist(normal_data, bins=40, alpha=0.6, color='#38ef7d', label='Normal', edgecolor='none')
    axes[i].hist(failure_data, bins=40, alpha=0.6, color='#f5576c', label='Failure', edgecolor='none')
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=11, fontweight='bold', color='#e4e7f1')
    axes[i].legend(fontsize=9)

plt.suptitle('Feature Distributions: Normal vs Failure', fontsize=16, fontweight='bold', color='#e4e7f1', y=1.02)
plt.tight_layout()
plt.savefig('data/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = df[features + ['failure']].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8},
            annot_kws={'size': 9, 'color': '#e4e7f1'})

ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=20, color='#e4e7f1')
plt.tight_layout()
plt.savefig('data/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Box plots for key features
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
key_features = ['temperature_C', 'vibration_mm_s', 'pressure_bar', 'power_kW']

for i, feature in enumerate(key_features):
    bp = df.boxplot(column=feature, by='failure', ax=axes[i],
                    patch_artist=True, return_type='dict',
                    boxprops=dict(linewidth=1.5),
                    medianprops=dict(color='#fbbf24', linewidth=2))
    
    colors_box = ['#38ef7d', '#f5576c']
    for j, box in enumerate(bp[feature]['boxes']):
        box.set_facecolor(colors_box[j])
        box.set_alpha(0.4)
    
    axes[i].set_title(feature.replace('_', ' ').title(), fontsize=11, fontweight='bold', color='#e4e7f1')
    axes[i].set_xlabel('Class (0=Normal, 1=Failure)', color='#8892a8')

plt.suptitle('Box Plots: Key Features by Class', fontsize=14, fontweight='bold', color='#e4e7f1')
plt.tight_layout()
plt.savefig('data/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 4️⃣ Data Preprocessing

In [ ]:
# Handle missing values
print("Before cleaning:")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Total samples: {len(df)}")

df_clean = df.copy()

# Fill missing values with median
for col in features:
    median_val = df_clean[col].median()
    df_clean[col].fillna(median_val, inplace=True)

print("\nAfter cleaning:")
print(f"  Missing values: {df_clean.isnull().sum().sum()}")
print(f"  Total samples: {len(df_clean)}")

# Feature scaling
scaler = StandardScaler()
X = df_clean[features].values
y = df_clean['failure'].values

X_scaled = scaler.fit_transform(X)

print(f"\n📐 Feature matrix shape: {X_scaled.shape}")
print(f"📐 Target vector shape: {y.shape}")
print(f"\n⚖️ Class balance: {np.bincount(y)} (Normal / Failure)")

In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"🏋️ Training set: {X_train.shape[0]} samples")
print(f"🧪 Test set: {X_test.shape[0]} samples")
print(f"\n📊 Train class distribution: {np.bincount(y_train)}")
print(f"📊 Test class distribution: {np.bincount(y_test)}")

## 5️⃣ Model Training & Comparison

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=12, min_samples_split=5, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, max_depth=6, learning_rate=0.1, random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"🤖 Training: {name}")
    print(f"{'='*60}")
    
    # Train
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    # Cross-validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='f1')
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC AUC': auc,
        'CV F1 (mean)': cv_scores.mean(),
        'CV F1 (std)': cv_scores.std()
    })
    
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  ROC AUC:   {auc:.4f}")
    print(f"  CV F1:     {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# Results table
results_df = pd.DataFrame(results)
print(f"\n{'='*80}")
print("📊 MODEL COMPARISON RESULTS")
print(f"{'='*80}")
results_df

In [ ]:
# Visual comparison of models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Metrics comparison bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
x = np.arange(len(metrics_to_plot))
width = 0.18
model_colors = ['#667eea', '#38ef7d', '#f5576c', '#fbbf24']

for i, (_, row) in enumerate(results_df.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    axes[0].bar(x + i * width, values, width, label=row['Model'],
                color=model_colors[i], alpha=0.85, edgecolor='white', linewidth=0.5)

axes[0].set_xlabel('Metric')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Performance Comparison', fontsize=13, fontweight='bold', color='#e4e7f1')
axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend(fontsize=9, loc='lower right')
axes[0].set_ylim(0.8, 1.02)

# ROC Curves
for i, (name, model) in enumerate(trained_models.items()):
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    axes[1].plot(fpr, tpr, color=model_colors[i], linewidth=2,
                 label=f'{name} (AUC={auc:.4f})')

axes[1].plot([0, 1], [0, 1], 'r--', alpha=0.5, linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves Comparison', fontsize=13, fontweight='bold', color='#e4e7f1')
axes[1].legend(fontsize=9, loc='lower right')
axes[1].set_xlim(-0.02, 1.02)
axes[1].set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig('data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6️⃣ Best Model: Detailed Analysis

In [ ]:
# Select best model (Random Forest)
best_model = trained_models['Random Forest']
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]

print("🏆 BEST MODEL: Random Forest")
print("=" * 50)
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=['Normal', 'Failure']))

In [ ]:
# Confusion Matrix Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Normal', 'Failure'],
            yticklabels=['Normal', 'Failure'],
            annot_kws={'size': 18, 'fontweight': 'bold'},
            linewidths=1, linecolor='#1a1f35')
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title('Confusion Matrix — Random Forest', fontsize=13, fontweight='bold', color='#e4e7f1')

# Feature Importance
importances = best_model.feature_importances_
sorted_idx = np.argsort(importances)
feature_labels = [features[i].replace('_', ' ').title() for i in sorted_idx]

colors_fi = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(features)))
axes[1].barh(feature_labels, importances[sorted_idx], color=colors_fi, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold', color='#e4e7f1')

for i, v in enumerate(importances[sorted_idx]):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=10, color='#e4e7f1')

plt.tight_layout()
plt.savefig('data/best_model_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 7️⃣ Real-Time Prediction Simulation

In [ ]:
from models.predictor import PumpFailurePredictor

# Initialize and train
predictor = PumpFailurePredictor()
metrics = predictor.train(df, model_name='random_forest')

print(f"✅ Model trained: {metrics['model_name']}")
print(f"   Test Accuracy: {metrics['test_accuracy']:.4f}")
print(f"   F1 Score: {metrics['f1_score']:.4f}")
print(f"   ROC AUC: {metrics['roc_auc']:.4f}")

In [ ]:
# Simulate Normal Operation
normal_reading = {
    'temperature_C': 63.5,
    'vibration_mm_s': 2.1,
    'pressure_bar': 3.6,
    'flow_rate_L_min': 118.5,
    'rpm': 1460,
    'power_kW': 14.8,
    'humidity_pct': 43.2,
    'noise_level_dB': 71.5
}

result = predictor.predict(normal_reading)
print("🟢 NORMAL OPERATION TEST")
print(f"   Status: {result['status']}")
print(f"   Risk Level: {result['risk_level']}")
print(f"   Failure Probability: {result['failure_probability']:.4f}")
print(f"   Alert: {result['alert']}")

In [ ]:
# Simulate Failure Condition
failure_reading = {
    'temperature_C': 98.2,
    'vibration_mm_s': 8.5,
    'pressure_bar': 1.5,
    'flow_rate_L_min': 62.3,
    'rpm': 1150,
    'power_kW': 24.1,
    'humidity_pct': 68.5,
    'noise_level_dB': 96.2
}

result = predictor.predict(failure_reading)
print("🔴 FAILURE CONDITION TEST")
print(f"   Status: {result['status']}")
print(f"   Risk Level: {result['risk_level']}")
print(f"   Failure Probability: {result['failure_probability']:.4f}")
print(f"   Alert: {'⚠️ YES — ALERT TRIGGERED!' if result['alert'] else 'No'}")

In [ ]:
# Batch prediction with visualization
np.random.seed(99)
n_sim = 50

sim_readings = []
for i in range(n_sim):
    is_failure = np.random.random() < 0.2
    if is_failure:
        reading = {
            'temperature_C': np.random.normal(92, 8),
            'vibration_mm_s': np.random.normal(7, 1.5),
            'pressure_bar': np.random.normal(2.0, 0.4),
            'flow_rate_L_min': np.random.normal(75, 12),
            'rpm': np.random.normal(1220, 60),
            'power_kW': np.random.normal(21, 2.5),
            'humidity_pct': np.random.normal(62, 8),
            'noise_level_dB': np.random.normal(90, 5)
        }
    else:
        reading = {
            'temperature_C': np.random.normal(65, 5),
            'vibration_mm_s': np.random.normal(2.5, 0.8),
            'pressure_bar': np.random.normal(3.5, 0.3),
            'flow_rate_L_min': np.random.normal(120, 10),
            'rpm': np.random.normal(1450, 30),
            'power_kW': np.random.normal(15, 1.5),
            'humidity_pct': np.random.normal(45, 8),
            'noise_level_dB': np.random.normal(72, 4)
        }
    sim_readings.append(reading)

# Run predictions
predictions = [predictor.predict(r) for r in sim_readings]
probs = [p['failure_probability'] for p in predictions]
statuses = [p['status'] for p in predictions]

# Visualize
fig, ax = plt.subplots(figsize=(16, 6))
colors_sim = ['#f5576c' if p > 0.5 else '#fbbf24' if p > 0.3 else '#38ef7d' for p in probs]
ax.bar(range(n_sim), probs, color=colors_sim, edgecolor='none', alpha=0.85)
ax.axhline(y=0.5, color='#f5576c', linestyle='--', linewidth=1.5, alpha=0.7, label='Failure Threshold')
ax.axhline(y=0.3, color='#fbbf24', linestyle='--', linewidth=1, alpha=0.5, label='Warning Threshold')
ax.set_xlabel('Reading #', fontsize=12)
ax.set_ylabel('Failure Probability', fontsize=12)
ax.set_title('Batch Prediction Simulation (50 Readings)', fontsize=14, fontweight='bold', color='#e4e7f1')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('data/batch_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
alerts_triggered = sum(1 for p in predictions if p['alert'])
print(f"\n📊 Simulation Summary:")
print(f"   Total Readings: {n_sim}")
print(f"   Alerts Triggered: {alerts_triggered}")
print(f"   Normal: {sum(1 for p in predictions if p['status'] == 'Normal')}")
print(f"   Warning: {sum(1 for p in predictions if p['status'] == 'Warning')}")
print(f"   Failure Risk: {sum(1 for p in predictions if p['status'] == 'Failure Risk')}")
print(f"   Imminent Failure: {sum(1 for p in predictions if p['status'] == 'Imminent Failure')}")

## 8️⃣ Save Model

In [ ]:
# Save the trained model
predictor.save_model()
print("\n✅ Model saved successfully!")
print("\n🚀 To launch the web dashboard, run:")
print("   python app.py")
print("   Then open http://localhost:5000")

---

## 📋 Summary

| Component | Details |
|---|---|
| **Dataset** | 5000 synthetic pump sensor readings (8 features) |
| **Models Trained** | Logistic Regression, Random Forest, SVM, Gradient Boosting |
| **Best Model** | Random Forest (highest F1 & ROC AUC) |
| **Preprocessing** | Median imputation, StandardScaler normalization |
| **Validation** | 80/20 train-test split + 5-fold stratified cross-validation |
| **Output** | Binary classification (Normal / Failure) + probability + risk level |
| **Alert System** | Threshold-based alerts (Low/Medium/High/Critical risk) |
| **Dashboard** | Flask web app at http://localhost:5000 |

### 🔜 Next Steps
- Integrate with IoT sensors (Arduino/Raspberry Pi) for real-time data
- Add cloud deployment (AWS/Azure) for remote monitoring
- Implement time-series forecasting for trend prediction
- Add anomaly detection for unsupervised failure detection